# Olist Orders — Heatmap and Funnel Chart of the Logistics Process

## Introduction

This analysis explores the orders dataset of **Olist**, the largest e-commerce platform in Brazil. Through interactive visualizations the following questions are answered:

- On which days of the week are orders most concentrated by status?
- How many orders complete each stage of the logistics process?
- At which stage do the greatest losses occur?

**Tools used:** Python · Pandas · Plotly  
**Dataset:** [Brazilian E-Commerce Public Dataset — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## 1. Data Preparation

### 1.1 Dataset Selection

The **Brazilian E-Commerce Public Dataset by Olist** was used, available on Kaggle: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

The file `olist_orders_dataset.csv` contains order information including purchase date, approval date, delivery dates and order status.

### 1.2 Initial Data Loading and Exploration

The file **olist_orders_dataset.csv** was loaded into the Python notebook using the pandas library. Once loaded, an initial exploration was performed to understand the structure, dimensions and data types of the dataset.

### 1.3 Data Preprocessing

To prepare the data for visualization, a series of basic preprocessing steps were applied.

First, the date/time columns were converted to datetime format to enable time-based operations. Then, the day of the week was extracted from the purchase timestamp to enable day-based analysis.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv("olist_orders_dataset.csv")

# Convert time columns to datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## 2. Visualizations

### 2.1 Heatmap with px.density_heatmap

Heatmap: day_of_week vs order_status (intensity = number of orders)

In [3]:
df_heat = df.dropna(subset=["order_purchase_timestamp", "order_status"]).copy()

df_heat["day_of_week"] = df_heat["order_purchase_timestamp"].dt.day_name()

# Order days Monday to Sunday
order_days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df_heat["day_of_week"] = pd.Categorical(df_heat["day_of_week"], categories=order_days, ordered=True)

heat_data = (
    df_heat.groupby(["day_of_week", "order_status"])
    .size()
    .reset_index(name="orders_count")
)
heat_data.head()

fig_heat = px.density_heatmap(
    heat_data,
    x="day_of_week",
    y="order_status",
    z="orders_count",
    color_continuous_scale="Blues",
    title="Heatmap: Number of Orders by Day of Week and Order Status"
)

fig_heat.update_layout(xaxis_title="Day of the Week", yaxis_title="Order Status")
fig_heat.show()
fig_heat.write_image("heatmap_pedidos_todos.png")

![Heatmap all statuses](heatmap_pedidos_todos.png)

To improve the visual interpretation of the heatmap, the most relevant order statuses within the main process flow are filtered (approved, shipped, delivered and canceled). This decision allows focusing on the stages that actually reflect the logistics cycle, reducing visual noise from less representative statuses.

In [4]:
valid_status = [
    "approved",
    "shipped",
    "delivered",
    "canceled"
]

heat_data_filtered = heat_data[
    heat_data["order_status"].isin(valid_status)
].copy()

status_order = ["approved", "shipped", "delivered", "canceled"]

heat_data_filtered["order_status"] = pd.Categorical(
    heat_data_filtered["order_status"],
    categories=status_order,
    ordered=True
)

fig_heat = px.density_heatmap(
    heat_data_filtered,
    x="day_of_week",
    y="order_status",
    z="orders_count",
    color_continuous_scale="Blues",
    title="Heatmap: Number of Orders by Day of Week and Order Status (Filtered)"
)

fig_heat.update_layout(
    xaxis_title="Day of the Week",
    yaxis_title="Order Status"
)

fig_heat.show()
fig_heat.write_image("heatmap_pedidos_filtrado.png")

![Heatmap filtered statuses](heatmap_pedidos_filtrado.png)

The predominance of the delivered status reflects that the majority of orders successfully complete the process, while intermediate or cancellation statuses show significantly lower frequency. This pattern is consistent across all days of the week, suggesting a stable and predictable logistics operation.

### 2.2 Funnel Chart

The number of orders reaching each stage of the process is measured:

* Purchased (order created) with **order_purchase_timestamp**
* Approved (payment approved) with **order_approved_at**
* Handed to Carrier (shipped to logistics operator) with **order_delivered_carrier_date**
* Delivered to Customer with **order_delivered_customer_date**

In [5]:
# Create stage indicators
df_fun = df.copy()

df_fun["stage_purchased"] = df_fun["order_purchase_timestamp"].notna()
df_fun["stage_approved"] = df_fun["order_approved_at"].notna()
df_fun["stage_handed_to_carrier"] = df_fun["order_delivered_carrier_date"].notna()
df_fun["stage_delivered"] = df_fun["order_delivered_customer_date"].notna()

# Count orders per stage
funnel_data = pd.DataFrame({
    "stage": ["Purchased", "Approved", "Handed to Carrier", "Delivered to Customer"],
    "orders_count": [
        df_fun["stage_purchased"].sum(),
        df_fun["stage_approved"].sum(),
        df_fun["stage_handed_to_carrier"].sum(),
        df_fun["stage_delivered"].sum(),
    ]
})

# % vs start and % vs previous stage
funnel_data["conversion_pct_vs_start"] = (
    funnel_data["orders_count"] / funnel_data.loc[0, "orders_count"] * 100
).round(2)

funnel_data["conversion_pct_vs_prev"] = (
    funnel_data["orders_count"].div(funnel_data["orders_count"].shift(1)) * 100
).round(2)

# Labels to display inside the funnel
funnel_data["label"] = (
    funnel_data["orders_count"].astype(str)
    + " ("
    + funnel_data["conversion_pct_vs_start"].astype(str)
    + "%)"
)

# Create chart with Plotly Express
fig_funnel = px.funnel(
    funnel_data,
    x="orders_count",
    y="stage",
    text="label",
    title="Funnel: Order Progression Through Logistics Process Stages",
    color_discrete_sequence=px.colors.sequential.Blues
)

fig_funnel.update_traces(textposition="inside")
fig_funnel.update_layout(
    xaxis_title="Number of Orders",
    yaxis_title="Stage"
)

fig_funnel.show()
fig_funnel.write_image("funnel_pedidos.png")
# Display the table
funnel_data

,stage,orders_count,conversion_pct_vs_start,conversion_pct_vs_prev,label
0,Purchased,99441,100.00,NaN,99441 (100.0%)
1,Approved,99281,99.84,99.84,99281 (99.84%)
2,Handed to Carrier,97658,98.21,98.37,97658 (98.21%)
3,Delivered to Customer,96476,97.02,98.79,96476 (97.02%)


![Funnel orders](funnel_pedidos.png)

The funnel chart shows the progressive reduction in the number of orders as they advance through the process stages: order created, payment approved, handed to logistics operator and final delivery to customer.

## 3. Conclusions

**Heatmap**

The heatmap shows that the majority of orders concentrate in the delivered status throughout all days of the week, indicating a stable logistics process. No significant variations by day are observed, suggesting a homogeneous temporal distribution of orders.

The heatmap is appropriate because it allows simultaneously comparing order status and day of the week, using color intensity to represent the number of orders. This visual encoding facilitates quick identification of concentration patterns and temporal stability.

**Funnel Chart**

The funnel shows a progressive reduction of orders between the different stages of the logistics process, with a high conversion rate between purchase and payment approval, and greater loss at the shipping stage. The majority of orders that are dispatched successfully complete with delivery to the customer.

The funnel chart is effective because it clearly represents a sequential conversion process, allowing quick identification of which stages produce the main losses and evaluating the efficiency of the logistics flow.

---
*Analysis by Juan Bautista Acuña — [GitHub](https://github.com/bautistaacuna)*